# CC Media Studio · Video Preview
选择 GPU T4 ×2，开启 Internet，然后运行全部单元格。 / Select GPU T4 ×2, enable Internet, and run all cells.
首次自动下载模型；后续请求复用当前模型。MP4 保存到 `/kaggle/working/cc-media-video.mp4`。 / First use downloads weights; subsequent requests reuse the active model.
结束后在 Kaggle 停止 GPU 会话。 / Stop the Kaggle GPU session when finished.

In [ ]:
import sys,json
from pathlib import Path
modules=Path('/kaggle/temp/cc-video-modules');modules.mkdir(exist_ok=True)
for name,source in {'asset_manager.py': '"""Download public HF weights with resumable transfers and real byte progress."""\nimport hashlib\nimport json\nimport os\nimport shutil\nimport time\nfrom pathlib import Path, PurePosixPath\nfrom urllib.parse import quote\nimport requests\n\nBASE_REPO = \'Qwen/Qwen-Image-2.1\'\nGGUF_REPO = \'azukivc/Qwen-Image-2.1-GGUF\'\nBASE_REVISION = \'d26bb61231c349cf6b7896fa83353113880e1ba3\'\nGGUF_REVISION = \'4e43fd824763d3d24233630c343462592f68ca49\'\n\n\ndef emit_progress(phase, message, done=None, total=None, filename=None):\n    event = {\'phase\': phase, \'message\': message}\n    if done is not None: event[\'downloaded_bytes\'] = done\n    if total: event.update(total_bytes=total, percent=round(min(100, 100*done/total), 1))\n    if filename: event[\'file\'] = filename\n    callback = globals().get(\'_QWEN_PROGRESS_CALLBACK\')\n    if callback: callback(event)\n    print(\'COMFY_PROGRESS:\' + json.dumps(event, ensure_ascii=False), flush=True)\n\n\ndef exception_chain(error, limit=6):\n    """\'Type: message <- Cause: message\' for an exception and what it wraps.\n\n    Libraries such as diffusers re-raise a generic error; the original is\n    only kept as __cause__/__context__."""\n    parts, seen = [], set()\n    while error is not None and id(error) not in seen and len(parts) < limit:\n        seen.add(id(error)); parts.append(type(error).__name__ + \': \' + str(error)[:300])\n        error = error.__cause__ or error.__context__\n    return \' <- \'.join(parts)\n\n\ndef file_matches_receipt(path):\n    """True/False when a download receipt exists, None when it cannot be checked."""\n    path = Path(path)\n    try:\n        receipt = json.loads(path.with_name(path.name + \'.verified.json\').read_text())\n    except (OSError, ValueError):\n        return None\n    if not path.is_file() or path.stat().st_size != receipt.get(\'size\'): return False\n    if not receipt.get(\'sha256\'): return None\n    digest = hashlib.sha256()\n    with path.open(\'rb\') as content:\n        for chunk in iter(lambda: content.read(16*1024*1024), b\'\'): digest.update(chunk)\n    return digest.hexdigest() == receipt[\'sha256\']\n\n\n# Model folders on the session\'s temporary disk. Kaggle sessions are capped\n# below what the host reports as free, so keep downloaded models within a\n# budget by removing the least recently used folders that are not in use.\nMODEL_ROOTS = (\'/kaggle/temp/cc-media-models\', \'/kaggle/temp/cc-media-video\', \'/kaggle/temp/qwen-image-assets\')\nDISK_BUDGET_BYTES = 80 * 1024**3\n\n\ndef _model_folders():\n    folders = []\n    for root in map(Path, MODEL_ROOTS):\n        if root.name == \'qwen-image-assets\':\n            if root.is_dir(): folders.append(root)\n        elif root.is_dir():\n            folders.extend(child for child in root.iterdir() if child.is_dir() and child.name != \'_shared\')\n    return folders\n\n\ndef _disk_bytes(paths):\n    """Bytes used by files under `paths`, counting hardlinked files once."""\n    seen, total = set(), 0\n    for base in paths:\n        for path in Path(base).rglob(\'*\'):\n            try:\n                stat = path.lstat()\n            except OSError:\n                continue\n            if path.is_file() and (stat.st_dev, stat.st_ino) not in seen:\n                # st_blocks (allocated space) is POSIX-only; fall back to the size.\n                blocks = getattr(stat, \'st_blocks\', None)\n                seen.add((stat.st_dev, stat.st_ino)); total += blocks * 512 if blocks is not None else stat.st_size\n    return total\n\n\ndef release_model_disk(keep, needed_bytes, budget=DISK_BUDGET_BYTES, notify=None, count_existing=True):\n    """Remove least recently used model folders until `needed_bytes` fits.\n\n    `keep` is the folder about to be used; it is never removed. Video files\n    shared through hardlinks in `_shared` are released once no model links them."""\n    keep = Path(keep).resolve()\n    # Files already in the target folder (a resumed or cached download) are not\n    # new. A copy of the folder (count_existing=False) is entirely new space.\n    if count_existing:\n        needed_bytes = max(0, needed_bytes - (_disk_bytes([keep]) if keep.is_dir() else 0))\n    shared = [Path(root)/\'_shared\' for root in MODEL_ROOTS if (Path(root)/\'_shared\').is_dir()]\n    used = _disk_bytes([*_model_folders(), *shared])\n    candidates = sorted((f for f in _model_folders() if f.resolve() != keep), key=lambda f: f.stat().st_mtime)\n    removed = []\n    for folder in candidates:\n        if used + needed_bytes <= budget: break\n        shutil.rmtree(folder, ignore_errors=True); removed.append(folder.name)\n        for link_root in shared:\n            for path in link_root.iterdir():\n                try:\n                    if path.is_file() and path.stat().st_nlink == 1: path.unlink()\n                except OSError: pass\n        used = _disk_bytes([*_model_folders(), *shared])\n    if removed and notify: notify(removed)\n    return removed\n\n\ndef repository_manifest(repo, session=None, revision=None):\n    http = session or requests.Session()\n    endpoint = \'https://huggingface.co/api/models/\' + repo\n    if revision: endpoint += \'/revision/\' + revision\n    response = http.get(endpoint, params={\'blobs\':\'true\'}, timeout=(20,60))\n    response.raise_for_status()\n    data = response.json()\n    if data.get(\'private\') or data.get(\'gated\'):\n        raise RuntimeError(\'默认权重必须能够公开下载，当前来源需要额外授权。\')\n    return data\n\n\ndef safe_relative(name):\n    path = PurePosixPath(name)\n    if path.is_absolute() or \'..\' in path.parts or \'\\\\\' in name:\n        raise ValueError(\'Invalid model file path\')\n    return path\n\n\ndef download_file(http, url, destination, expected_size, expected_sha=None, progress=None, verify_progress=None):\n    destination = Path(destination); destination.parent.mkdir(parents=True, exist_ok=True)\n    partial = destination.with_name(destination.name + \'.partial\')\n    receipt = destination.with_name(destination.name + \'.verified.json\')\n    if destination.is_file() and destination.stat().st_size == expected_size:\n        try:\n            stamp = json.loads(receipt.read_text())\n            if stamp == {\'size\':expected_size, \'sha256\':expected_sha}:\n                if progress: progress(expected_size)\n                return\n        except (OSError,ValueError): pass\n        # Existing files without a verification receipt must still be checked.\n        partial.unlink(missing_ok=True); destination.replace(partial)\n    for attempt in range(3):\n        try:\n            offset = partial.stat().st_size if partial.exists() else 0\n            if expected_size and offset > expected_size:\n                partial.unlink(); offset = 0\n            if offset != expected_size or not partial.exists():\n                headers = {\'Range\':f\'bytes={offset}-\'} if offset else {}\n                with http.get(url, headers=headers, stream=True, timeout=(20,60)) as response:\n                    response.raise_for_status()\n                    if offset and response.status_code == 206:\n                        if not response.headers.get(\'Content-Range\',\'\').startswith(f\'bytes {offset}-\'):\n                            raise ValueError(\'Invalid download resume range\')\n                    else:\n                        offset = 0\n                    with partial.open(\'ab\' if offset else \'wb\') as output:\n                        for chunk in response.iter_content(chunk_size=4*1024*1024):\n                            if chunk:\n                                output.write(chunk); offset += len(chunk)\n                                if progress: progress(offset)\n            if expected_size and partial.stat().st_size != expected_size:\n                raise OSError(\'Incomplete model download\')\n            if expected_sha:\n                digest = hashlib.sha256()\n                with partial.open(\'rb\') as content:\n                    verified = 0\n                    for chunk in iter(lambda:content.read(16*1024*1024), b\'\'):\n                        digest.update(chunk); verified += len(chunk)\n                        if verify_progress: verify_progress(verified)\n                if digest.hexdigest() != expected_sha:\n                    partial.unlink(); raise ValueError(\'Model checksum mismatch\')\n            partial.replace(destination)\n            receipt.write_text(json.dumps({\'size\':expected_size,\'sha256\':expected_sha}))\n            if progress: progress(destination.stat().st_size)\n            return\n        except (requests.RequestException, OSError, ValueError):\n            if attempt == 2: raise\n            time.sleep(2*(attempt+1))\n\n\ndef prepare_hf_assets(folder, http=None):\n    folder = Path(folder); folder.mkdir(parents=True, exist_ok=True)\n    http = http or requests.Session()\n    emit_progress(\'manifest\', \'正在从 Hugging Face 获取模型文件清单。\')\n    base = repository_manifest(BASE_REPO, http, BASE_REVISION)\n    quant = repository_manifest(GGUF_REPO, http, GGUF_REVISION)\n    records = []\n    for item in base[\'siblings\']:\n        name = item[\'rfilename\']\n        if name in {\'model_index.json\',\'LICENSE\',\'transformer/config.json\'} or any(name.startswith(p+\'/\') for p in [\'processor\',\'scheduler\',\'vae\',\'text_encoder\']):\n            if name.endswith(\'.json\') or name.endswith(\'.safetensors\') or name.endswith(\'.txt\') or name.endswith(\'.model\') or name.endswith(\'.jinja\') or name.endswith(\'.tiktoken\') or name == \'LICENSE\':\n                records.append((BASE_REPO,base[\'sha\'],item,name))\n    candidates = [f for f in quant[\'siblings\'] if f[\'rfilename\'].lower().endswith(\'q4_k_m.gguf\')]\n    if len(candidates) != 1: raise RuntimeError(\'兼容的 Q4_K_M 权重清单不明确。\')\n    records.append((GGUF_REPO,quant[\'sha\'],candidates[0],\'transformer-q4.gguf\'))\n    if not any(name.startswith(\'text_encoder/\') and name.endswith(\'.safetensors\') for _,_,_,name in records):\n        raise RuntimeError(\'模型清单缺少文本编码器权重。\')\n    total = sum(int(item.get(\'size\') or item.get(\'lfs\',{}).get(\'size\') or 0) for _,_,item,_ in records)\n    cached=True\n    for _,_,item,name in records:\n        safe_relative(name)\n        size=int(item.get(\'size\') or item.get(\'lfs\',{}).get(\'size\') or 0)\n        target=folder/name\n        try:\n            receipt=json.loads(target.with_name(target.name+\'.verified.json\').read_text())\n            if not target.is_file() or target.stat().st_size!=size or receipt!={\'size\':size,\'sha256\':item.get(\'lfs\',{}).get(\'sha256\')}:\n                cached=False;break\n        except (OSError,ValueError):cached=False;break\n    if cached:\n        emit_progress(\'cached\',\'已找到完整模型缓存，无需重新下载。\',total,total or None)\n        return folder\n    release_model_disk(folder, total, notify=lambda names: emit_progress(\'cleanup\', \'正在清理较早使用的模型文件以释放磁盘：\'+\'、\'.join(names)))\n    done = 0; last = [0.0]\n    for repo,revision,item,name in records:\n        safe_relative(name)\n        size = int(item.get(\'size\') or item.get(\'lfs\',{}).get(\'size\') or 0)\n        sha = item.get(\'lfs\',{}).get(\'sha256\')\n        start = done\n        def progress(value, name=name, start=start):\n            now=time.monotonic()\n            if now-last[0]>=.8 or value==size:\n                emit_progress(\'download\',\'首次准备：正在下载模型权重。\',start+value,total or None,name)\n                last[0]=now\n        url=\'https://huggingface.co/\'+repo+\'/resolve/\'+revision+\'/\'+quote(item[\'rfilename\'],safe=\'/\')\n        def verify_progress(value, name=name, size=size):\n            now=time.monotonic()\n            if now-last[0]>=.8 or value==size:\n                emit_progress(\'verify\',\'正在校验下载的权重。\',value,size,name);last[0]=now\n        download_file(http,url,folder/name,size,sha,progress,verify_progress)\n        done += size\n    (folder/\'asset-sources.json\').write_text(json.dumps({\'base_repo\':BASE_REPO,\'base_revision\':base[\'sha\'],\'gguf_repo\':GGUF_REPO,\'gguf_revision\':quant[\'sha\'],\'download_bytes\':total},indent=2))\n    emit_progress(\'download\',\'模型权重下载完成。\',done,total or None)\n    return folder\n\n\ndef copy_assets(source, target):\n    source,target=Path(source),Path(target)\n    files=[p for p in source.rglob(\'*\') if p.is_file()]\n    total=sum(p.stat().st_size for p in files); done=0; last=0\n    for path in files:\n        destination=target/path.relative_to(source); destination.parent.mkdir(parents=True,exist_ok=True)\n        with path.open(\'rb\') as src, destination.open(\'wb\') as dst:\n            for chunk in iter(lambda:src.read(8*1024*1024),b\'\'):\n                dst.write(chunk);done+=len(chunk)\n                now=time.monotonic()\n                if now-last>=.8:\n                    emit_progress(\'copy\',\'正在准备已缓存的模型文件。\',done,total,path.name);last=now\n    emit_progress(\'copy\',\'模型文件准备完成。\',done,total)\n', 'cache_manager.py': '"""Private, create-only Kaggle cache of prepared INT8/Q4 model assets.\n\nExecuted in the user\'s Kaggle kernel. Credentials are passed at connection time,\nused only for Kaggle operations, and removed by the caller\'s finally block.\n"""\nimport contextlib\nimport tempfile\nimport threading\nfrom concurrent.futures import ThreadPoolExecutor\n\nCACHE_VERSION = 1\n\n\ndef cache_file_hashes(folder):\n    hashes = {}\n    for file in Path(folder).rglob(\'*\'):\n        if file.is_file():\n            with file.open(\'rb\') as content:\n                hashes[file.relative_to(folder).as_posix()] = hashlib.file_digest(content,\'sha256\').hexdigest()\n    return hashes\n\n\n@contextlib.contextmanager\ndef cache_credentials(handle, token):\n    owner = handle.split(\'/\')[0]\n    values = {\'KAGGLE_USERNAME\': owner,\'KAGGLE_API_TOKEN\':None,\'KAGGLE_KEY\':None}\n    if token.startswith(\'KGAT_\'): values[\'KAGGLE_API_TOKEN\'] = token\n    else: values[\'KAGGLE_KEY\'] = token\n    old = {name: os.environ.get(name) for name in values}\n    try:\n        for name,value in values.items():\n            if value is None: os.environ.pop(name,None)\n            else: os.environ[name]=value\n        yield\n    finally:\n        for name, value in old.items():\n            if value is None: os.environ.pop(name, None)\n            else: os.environ[name] = value\n\n\ndef cache_metadata(handle, token):\n    owner, slug = handle.split(\'/\')\n    with requests.Session() as http:\n        if token.startswith(\'KGAT_\'): http.headers[\'Authorization\'] = \'Bearer \' + token\n        else: http.auth = (owner, token)\n        response = http.get(\'https://www.kaggle.com/api/v1/datasets/view/\'+owner+\'/\'+slug, timeout=(20,60))\n        if response.status_code in (403,404):\n            # Kaggle hides missing private slugs behind 403. Verify absence in\n            # the authenticated owner listing; a permission failure stays fatal.\n            for page in range(1,101):\n                listing = http.get(\'https://www.kaggle.com/api/v1/datasets/list\',\n                    params={\'user\':owner,\'search\':slug,\'page\':page}, timeout=(20,60))\n                listing.raise_for_status()\n                records = listing.json()\n                if not isinstance(records,list): raise RuntimeError(\'Invalid dataset listing\')\n                if any(item.get(\'ref\') == handle for item in records):\n                    raise RuntimeError(\'Existing cache is inaccessible\')\n                if len(records)<20: return None\n            raise RuntimeError(\'Cannot confirm cache absence\')\n        response.raise_for_status()\n        data = response.json()\n        if data.get(\'isPrivate\') is not True:\n            raise RuntimeError(\'模型缓存必须为私有 Dataset；没有修改现有 Dataset。\')\n        return data\n\n\ndef validate_cache_manifest(manifest):\n    generic=manifest.get(\'studio_model_id\') is not None\n    if generic:\n        if (manifest.get(\'studio_cache_version\')!=2 or manifest[\'studio_model_id\']!=globals().get(\'_CC_MODEL_ID\')\n            or manifest.get(\'model_sources\')!=globals().get(\'_CC_MODEL_SOURCES\')):\n            raise RuntimeError(\'所选模型缓存版本不兼容。\')\n    elif (manifest.get(\'studio_cache_version\') != CACHE_VERSION\n        or manifest.get(\'base_revision\') != BASE_REVISION\n        or manifest.get(\'gguf_revision\') != GGUF_REVISION):\n        raise RuntimeError(\'同名 Dataset 不是兼容的本程序缓存，没有覆盖它。\')\n    files = manifest.get(\'cache_files\', {})\n    if not files or (not generic and \'transformer-q4.gguf\' not in files) or (generic and not any(name.endswith(\'.safetensors\') for name in files)):\n        raise RuntimeError(\'模型缓存文件清单不完整。\')\n    for name, size in files.items():\n        safe_relative(name)\n        if not isinstance(size,int) or size<0: raise RuntimeError(\'Invalid cached file size\')\n    return manifest\n\n\ndef validate_cache(folder):\n    folder = Path(folder)\n    manifest = validate_cache_manifest(json.loads((folder/\'asset-sources.json\').read_text()))\n    for name, size in manifest[\'cache_files\'].items():\n        file = folder/name\n        if not file.is_file() or file.stat().st_size != size:\n            raise RuntimeError(\'模型缓存文件不完整。\')\n    return manifest\n\n\ndef cache_http(handle, token):\n    http = requests.Session()\n    if token.startswith(\'KGAT_\'): http.headers[\'Authorization\'] = \'Bearer \'+token\n    else: http.auth = (handle.split(\'/\')[0],token)\n    return http\n\n\ndef cache_file_url(handle, version, name):\n    safe_relative(name)\n    return (\'https://www.kaggle.com/api/v1/datasets/download/\'+handle+\'/\'+quote(name,safe=\'\')+\n        \'?raw=true&dataset_version_number=\'+str(int(version)))\n\n\ndef read_cache_manifest(http, handle, version):\n    response = http.get(cache_file_url(handle,version,\'asset-sources.json\'),timeout=(20,60))\n    response.raise_for_status()\n    return validate_cache_manifest(response.json())\n\n\ndef download_cache_assets(folder, handle, token, metadata):\n    folder = Path(folder); folder.mkdir(parents=True,exist_ok=True)\n    version = metadata[\'currentVersionNumber\']\n    with cache_http(handle,token) as http:\n        manifest = read_cache_manifest(http,handle,version)\n    total = sum(manifest[\'cache_files\'].values()); received={}; last=[0.0]\n    progress_lock=threading.Lock()\n    def transfer(item):\n        name,size=item\n        def progress(value):\n            with progress_lock:\n                received[name]=value\n                now=time.monotonic()\n                if now-last[0]>=.8 or value==size:\n                    emit_progress(\'cache-download\',\'正在拉取私有 Dataset 模型缓存。\',sum(received.values()),total,name)\n                    last[0]=now\n        with cache_http(handle,token) as http:\n            download_file(http,cache_file_url(handle,version,name),folder/name,size,\n                manifest.get(\'cache_sha256\',{}).get(name),progress)\n    # Independent resumable streams avoid serializing all multi-GB shards.\n    files=sorted(manifest[\'cache_files\'].items(),key=lambda item:item[1],reverse=True)\n    with ThreadPoolExecutor(max_workers=4) as pool:\n        list(pool.map(transfer,files))\n    (folder/\'asset-sources.json\').write_text(json.dumps(manifest,indent=2))\n    validate_cache(folder)\n    return folder\n\n\ndef prepare_auto_assets(folder):\n    handle, token = globals().get(\'_QWEN_CACHE_HANDLE\'), globals().get(\'_QWEN_CACHE_TOKEN\')\n    globals()[\'_QWEN_CACHE_STATUS\'] = \'disabled\'\n    globals()[\'_QWEN_CACHE_STATUS_HANDLE\'] = handle\n    if handle and token:\n        try:\n            emit_progress(\'cache-check\', \'正在检查你的私有模型缓存。\')\n            metadata = cache_metadata(handle, token)\n            if metadata:\n                emit_progress(\'cache-download\', \'正在拉取私有 Dataset 模型缓存。\')\n                download_cache_assets(folder, handle, token, metadata)\n                globals()[\'_QWEN_CACHE_STATUS\'] = \'saved\'\n                globals()[\'_QWEN_RESOLVED_SOURCE\'] = \'kaggle-cache\'\n                return Path(folder), False\n            globals()[\'_QWEN_CACHE_STATUS\'] = \'missing\'\n        except Exception:\n            # A metadata/auth/compatibility error must never permit overwriting\n            # an existing dataset. Generation can still use the public weights.\n            globals()[\'_QWEN_CACHE_STATUS\'] = \'unavailable\'\n            emit_progress(\'cache-warning\', \'私有缓存暂不可用，本次从 Hugging Face 准备；没有修改已有 Dataset。\')\n    globals()[\'_QWEN_RESOLVED_SOURCE\'] = \'huggingface\'\n    return prepare_hf_assets(folder), True\n\n\ndef save_private_cache(folder, encoder):\n    handle, token = globals().get(\'_QWEN_CACHE_HANDLE\'), globals().get(\'_QWEN_CACHE_TOKEN\')\n    status = globals().get(\'_QWEN_CACHE_STATUS\', \'missing\') if globals().get(\'_QWEN_CACHE_STATUS_HANDLE\') == handle else \'missing\'\n    globals()[\'_QWEN_CACHE_STATUS_HANDLE\'] = handle\n    if not handle or not token or status in (\'saved\',\'unavailable\'): return\n    try:\n        emit_progress(\'cache-check\', \'正在确认私有缓存状态。\')\n        metadata = cache_metadata(handle, token)\n        if metadata:\n            with cache_http(handle,token) as http:\n                read_cache_manifest(http,handle,metadata[\'currentVersionNumber\'])\n            globals()[\'_QWEN_CACHE_STATUS\'] = \'saved\'\n            return\n        folder = Path(folder)\n        emit_progress(\'cache-prepare\', \'正在保存 INT8 编码器和 Q4 权重，首次需要额外等待。\')\n        generic=globals().get(\'_CC_MODEL_ID\',\'qwen-image-2.1\')!=\'qwen-image-2.1\'\n        # A bundle left by an interrupted save is never reused; remove it.\n        for stale in Path(\'/kaggle/temp\').glob(\'qwen-private-cache-*\'):\n            shutil.rmtree(stale, ignore_errors=True)\n        # Qwen\'s bundle copies its weights (other models use hardlinks), so the\n        # copy needs room within the model disk budget too.\n        if not generic and callable(globals().get(\'release_model_disk\')):\n            release_model_disk(folder, _disk_bytes([folder]), count_existing=False,\n                notify=lambda names: emit_progress(\'cleanup\', \'正在清理较早使用的模型文件以释放磁盘：\'+\'、\'.join(names)))\n        with tempfile.TemporaryDirectory(prefix=\'qwen-private-cache-\', dir=\'/kaggle/temp\') as temporary:\n            bundle = Path(temporary)\n            if generic:\n                def link_or_copy(src,dst):\n                    try:os.link(src,dst)\n                    except OSError:shutil.copy2(src,dst)\n                shutil.copytree(folder,bundle,dirs_exist_ok=True,copy_function=link_or_copy,\n                    ignore=shutil.ignore_patterns(\'*.partial\',\'*.verified.json\',\'asset-sources.json\'))\n            for name in (() if generic else (\'processor\',\'scheduler\',\'vae\',\'transformer\')):\n                shutil.copytree(folder/name, bundle/name, ignore=shutil.ignore_patterns(\'*.partial\',\'*.verified.json\'))\n            for name in (() if generic else (\'transformer-q4.gguf\',\'model_index.json\',\'LICENSE\')):\n                if (folder/name).exists(): shutil.copy2(folder/name, bundle/name)\n            if not generic:encoder.save_pretrained(bundle/\'text_encoder\', safe_serialization=True, max_shard_size=\'4GB\')\n            manifest = json.loads((folder/\'asset-sources.json\').read_text())\n            manifest.update(studio_cache_version=2 if generic else CACHE_VERSION,\n                cache_files={p.relative_to(bundle).as_posix():p.stat().st_size for p in bundle.rglob(\'*\') if p.is_file()},\n                cache_sha256=cache_file_hashes(bundle))\n            (bundle/\'asset-sources.json\').write_text(json.dumps(manifest, indent=2))\n            validate_cache(bundle)\n            emit_progress(\'cache-upload\', \'正在上传私有 Dataset；Kaggle 将显示文件上传进度。\')\n            # Use KaggleHub\'s create-only primitive: its public upload helper\n            # creates a new version on conflict, which could overwrite user data.\n            from kagglehub import datasets\n            api = datasets.dataset_upload.__globals__\n            create = datasets.create_dataset_or_version.__globals__[\'_create_dataset\']\n            with cache_credentials(handle, token):\n                uploads = api[\'upload_files_and_directories\'](str(bundle), item_type=api[\'ApiBlobType\'].DATASET,\n                    ignore_patterns=api[\'normalize_patterns\'](default=api[\'DEFAULT_IGNORE_PATTERNS\'], additional=None))\n                create(api[\'parse_dataset_handle\'](handle), uploads)\n            # The create primitive explicitly sends is_private=True. Confirm\n            # privacy again from the server before reporting success.\n            emit_progress(\'cache-processing\', \'文件已上传，Kaggle 正在处理私有 Dataset。\')\n            deadline = time.monotonic()+180\n            while not cache_metadata(handle, token):\n                if time.monotonic() >= deadline: raise RuntimeError(\'Dataset metadata not ready\')\n                time.sleep(5)\n        globals()[\'_QWEN_CACHE_STATUS\'] = \'saved\'\n        emit_progress(\'cache-saved\', \'私有模型缓存已保存，下次新会话优先拉取，无需重新下载或量化。\')\n    except Exception as error:\n        globals()[\'_QWEN_CACHE_STATUS\'] = \'failed\'\n        globals()[\'_QWEN_CACHE_FAILURE_TYPE\'] = type(error).__name__\n        emit_progress(\'cache-warning\', \'私有缓存保存未完成；本次仍可正常生图，重新连接后会再次检查。\')\n', 'video_precision.py': '"""Explicit device moves for INT8 video encoders without changing image helpers."""\n\n\ndef move_video_encoder(model,device):\n    import torch\n    from bitsandbytes.nn import Linear8bitLt\n    target=torch.device(device)\n    if all(p.device==target for p in model.parameters()):return model\n    layers=[]\n    for module in model.modules():\n        if not isinstance(module,Linear8bitLt):continue\n        scale=module.state.SCB if module.state.SCB is not None else module.weight.SCB\n        if scale is None:raise RuntimeError(\'Missing INT8 video scale\')\n        layers.append((module,scale))\n    for module,_ in layers:\n        module.state.CB=None;module.state.SCB=None;module.weight.CB=None;module.weight.SCB=None\n    try:\n        # Transformers forbids .to() on some 8-bit model classes even though\n        # torch/bitsandbytes support moving their quantized parameters.\n        return torch.nn.Module.to(model,target)\n    finally:\n        for module,scale in layers:\n            module.state.CB=module.weight.data;module.state.SCB=scale.to(module.weight.device)\n            module.weight.CB=None;module.weight.SCB=None\n', 'video_registry.py': '"""Video-only catalog; ordinary image clients do not import GPU packages."""\nVIDEO_MODELS = {\n    \'ltx-video-2b-distilled\': {\n        \'id\':\'ltx-video-2b-distilled\',\'name\':\'LTX-Video 2B · Efficient\',\'tier\':\'efficient\',\n        \'family\':\'ltx\',\'default_steps\':8,\'fixed_steps\':8,\'width\':768,\'height\':448,\n        \'frames\':33,\'fps\':16,\'guidance\':1.0,\'cache_slug\':\'cc-media-ltx-2b-distilled\',\'image_to_video\':True,\n        \'first_use\':{\'minutes\':\'7–11\',\'download_gb\':27},\n        \'repo\':\'Lightricks/LTX-Video\',\'revision\':\'8984fa25007f376c1a299016d0957a37a2f797bb\',\n        \'checkpoint\':\'ltxv-2b-0.9.6-distilled-04-25.safetensors\',\n        \'sigmas\':[1.0,.9937,.9875,.9812,.975,.9094,.725,.4219],\n        \'config_repo\':\'Lightricks/LTX-Video-0.9.5\',\'config_revision\':\'e58e28c39631af4d1468ee57a853764e11c1d37e\'},\n    \'wan21-t2v-1.3b\': {\n        \'id\':\'wan21-t2v-1.3b\',\'name\':\'Wan2.1 1.3B · Balanced\',\'tier\':\'balanced\',\n        \'family\':\'wan\',\'default_steps\':30,\'fixed_steps\':None,\'width\':832,\'height\':480,\n        \'frames\':33,\'fps\':16,\'guidance\':5.0,\'cache_slug\':\'cc-media-wan21-1-3b\',\'image_to_video\':False,\n        \'first_use\':None,\n        \'repo\':\'Wan-AI/Wan2.1-T2V-1.3B-Diffusers\',\'revision\':\'0fad780a534b6463e45facd96134c9f345acfa5b\'},\n    \'wan22-ti2v-5b\': {\n        \'id\':\'wan22-ti2v-5b\',\'name\':\'Wan2.2 5B · Quality\',\'tier\':\'quality\',\n        \'family\':\'wan22\',\'default_steps\':30,\'fixed_steps\':None,\'width\':768,\'height\':448,\n        \'frames\':33,\'fps\':24,\'guidance\':5.0,\'cache_slug\':\'cc-media-wan22-5b\',\'image_to_video\':True,\n        \'first_use\':{\'minutes\':\'19\',\'download_gb\':32},\n        \'repo\':\'Wan-AI/Wan2.2-TI2V-5B-Diffusers\',\'revision\':\'b8fff7315c768468a5333511427288870b2e9635\'},\n}\n\n\ndef video_profile(model_id=\'wan21-t2v-1.3b\'):\n    if not isinstance(model_id,str) or model_id not in VIDEO_MODELS:\n        raise ValueError(\'未知视频模型 / Unknown video model.\')\n    return dict(VIDEO_MODELS[model_id])\n\n\ndef video_catalog():\n    return [{k:v for k,v in p.items() if k not in (\'repo\',\'revision\',\'checkpoint\',\'config_repo\',\'config_revision\')}\n            for p in VIDEO_MODELS.values()]\n\n\ndef validate_video(prompt,seed=42,steps=None,model_id=\'wan21-t2v-1.3b\',frames=None,image=None,negative_prompt=None):\n    """Validate a video request. `image` is passed through unchanged: each layer\n    carries its own representation (base64, stored digest, bytes)."""\n    p=video_profile(model_id)\n    if not isinstance(prompt,str) or not 1<=len(prompt.strip())<=4000:\n        raise ValueError(\'提示词需要 1–4000 个字符。\')\n    if type(seed) is not int or not 0<=seed<=2147483647:raise ValueError(\'Invalid seed\')\n    steps=p[\'default_steps\'] if steps is None else steps\n    frames=p[\'frames\'] if frames is None else frames\n    if type(steps) is not int or not 1<=steps<=50:raise ValueError(\'steps must be 1–50\')\n    if p[\'fixed_steps\'] and steps!=p[\'fixed_steps\']:raise ValueError(\'LTX distilled requires 8 steps\')\n    if type(frames) is not int or not 9<=frames<=81 or (frames-1)%8:\n        raise ValueError(\'frames must be 9, 17, …, 81\')\n    result={\'prompt\':prompt,\'seed\':seed,\'steps\':steps,\'model_id\':p[\'id\'],\'frames\':frames}\n    if negative_prompt is not None and negative_prompt!=\'\':\n        if not isinstance(negative_prompt,str) or len(negative_prompt)>2000:raise ValueError(\'负面提示词最多 2000 字。\')\n        if p[\'guidance\']<=1:raise ValueError(p[\'name\']+\' 不使用负面提示词（无 CFG）。\')\n        if negative_prompt.strip():result[\'negative_prompt\']=negative_prompt\n    if image is not None and image!=\'\':\n        if not p[\'image_to_video\']:raise ValueError(p[\'name\']+\' 不支持图生视频；请选择 LTX-Video 2B 或 Wan2.2 5B。\')\n        result[\'image\']=image\n    return result\n', 'video_assets.py': '"""Pinned, resumable video downloads using the existing checked transfer helper."""\nimport json\nimport os\nimport shutil\nimport time\nfrom pathlib import Path\nfrom urllib.parse import quote\n\n\ndef video_asset_records(profile,manifest):\n    if manifest.get(\'sha\')!=profile[\'revision\']:raise RuntimeError(\'Video revision mismatch\')\n    records=[]\n    for item in manifest[\'siblings\']:\n        name=item[\'rfilename\']\n        from asset_manager import safe_relative\n        safe_relative(name)\n        parts=name.split(\'/\')\n        metadata=name in (\'model_index.json\',\'README.md\') or (\'license\' in name.lower() and len(parts)==1)\n        component=parts[0] in (\'text_encoder\',\'tokenizer\',\'transformer\',\'scheduler\',\'vae\')\n        keep=metadata or (component and name.endswith((\'.json\',\'.txt\',\'.model\',\'.safetensors\')))\n        if profile[\'family\']==\'ltx\':\n            keep=metadata or name==profile[\'checkpoint\'] or (component and\n                (not name.endswith(\'.safetensors\') or parts[0]==\'text_encoder\'))\n            if profile.get(\'config_repo\') and (name==\'model_index.json\' or parts[0] in (\'vae\',\'scheduler\',\'transformer\')):keep=False\n        if keep:records.append(item)\n    if not any(i[\'rfilename\'].endswith(\'.safetensors\') for i in records):raise RuntimeError(\'Missing video weights\')\n    return records\n\n\ndef prepare_video_assets(profile,folder,progress=None):\n    import requests\n    from asset_manager import repository_manifest,download_file\n    folder=Path(folder);folder.mkdir(parents=True,exist_ok=True)\n    notify=progress or (lambda event:None)\n    notify({\'phase\':\'manifest\'})\n    with requests.Session() as http:\n        records=[(profile[\'repo\'],profile[\'revision\'],item) for item in\n            video_asset_records(profile,repository_manifest(profile[\'repo\'],http,profile[\'revision\']))]\n        if profile.get(\'config_repo\'):\n            manifest=repository_manifest(profile[\'config_repo\'],http,profile[\'config_revision\'])\n            if manifest.get(\'sha\')!=profile[\'config_revision\']:raise RuntimeError(\'Video config revision mismatch\')\n            names={\'model_index.json\',\'vae/config.json\',\'transformer/config.json\',\'scheduler/scheduler_config.json\'}\n            records.extend((profile[\'config_repo\'],profile[\'config_revision\'],item) for item in manifest[\'siblings\'] if item[\'rfilename\'] in names)\n        sizes=[int(i.get(\'size\') or i.get(\'lfs\',{}).get(\'size\') or 0) for _,_,i in records]\n        if any(size<=0 for size in sizes):raise RuntimeError(\'Missing video file size\')\n        total=sum(sizes);done=0;last=[0.0]\n        from asset_manager import release_model_disk\n        release_model_disk(folder,total,notify=lambda names:notify({\'phase\':\'cleanup\',\'file\':\', \'.join(names)}))\n        for (repo,revision,item),size in zip(records,sizes):\n            name=item[\'rfilename\'];start=done\n            def report(value,start=start,name=name,size=size):\n                now=time.monotonic()\n                if now-last[0]>=1 or value==size:\n                    notify(dict(phase=\'download\',file=name,downloaded_bytes=start+value,\n                        total_bytes=total,percent=round(100*(start+value)/total,1)));last[0]=now\n            url=\'https://huggingface.co/\'+repo+\'/resolve/\'+revision+\'/\'+quote(name,safe=\'/\')\n            digest=item.get(\'lfs\',{}).get(\'sha256\');destination=folder/name\n            shared=folder.parent/\'_shared\'/(digest or \'unused\')\n            if digest and shared.is_file():\n                download_file(http,url,shared,size,digest)\n                destination.parent.mkdir(parents=True,exist_ok=True)\n                if not destination.exists():\n                    os.link(shared,destination)\n                    receipt=shared.with_name(shared.name+\'.verified.json\')\n                    shutil.copy2(receipt,destination.with_name(destination.name+\'.verified.json\'))\n                download_file(http,url,destination,size,digest,report)\n            else:\n                download_file(http,url,destination,size,digest,report)\n                if digest:\n                    shared.parent.mkdir(exist_ok=True)\n                    try:os.link(destination,shared)\n                    except FileExistsError:pass\n                    shutil.copy2(destination.with_name(destination.name+\'.verified.json\'),\n                        shared.with_name(shared.name+\'.verified.json\'))\n            done+=size\n    (folder/\'asset-sources.json\').write_text(json.dumps({\'studio_model_id\':profile[\'id\'],\n        \'model_sources\':video_sources(profile),\'download_bytes\':total}),encoding=\'utf-8\')\n    return folder\n\n\ndef video_cache_context(profile,handle,token,progress):\n    """Use the existing account-owned, create-only Dataset cache protocol."""\n    scope={\'_CC_MODEL_ID\':profile[\'id\'],\'_CC_MODEL_SOURCES\':video_sources(profile),\n        \'_QWEN_CACHE_HANDLE\':handle,\'_QWEN_CACHE_TOKEN\':token,\'_QWEN_PROGRESS_CALLBACK\':progress}\n    for name in (\'asset_manager.py\',\'cache_manager.py\'):\n        exec(compile(Path(__file__).with_name(name).read_text(encoding=\'utf-8\'),\'<video-cache>\',\'exec\'),scope)\n    return scope\n\n\ndef video_sources(profile):\n    sources=[[profile[\'repo\'],profile[\'revision\']]]\n    if profile.get(\'config_repo\'):sources.append([profile[\'config_repo\'],profile[\'config_revision\']])\n    return sources\n', 'video_parallel.py': '"""Transactional two-GPU CFG for Wan\'s separate conditional/unconditional calls."""\nfrom concurrent.futures import ThreadPoolExecutor\nfrom contextlib import nullcontext\nfrom functools import wraps\n\n\nclass WanParallelCFG:\n    def __init__(self,primary,secondary,negative,torch_module=None):\n        if torch_module is None:\n            import torch as torch_module\n        self.torch=torch_module;self.primary=primary;self.secondary=secondary\n        self.negative=negative.to(\'cuda:1\');self.pending=None;self.calls=0\n\n    def __enter__(self):\n        self.original=self.primary.forward\n        self.pool=ThreadPoolExecutor(max_workers=1,thread_name_prefix=\'cc-video-cfg\')\n        @wraps(self.original)\n        def forward(*args,**kwargs):\n            if args or kwargs.get(\'return_dict\') is not False:\n                raise ValueError(\'Unsupported Wan CFG forward signature\')\n            if self.pending is not None:\n                pending,self.pending=self.pending,None\n                output,event=pending.result();event.synchronize();self.calls+=1\n                return tuple(x.to(\'cuda:0\') if hasattr(x,\'to\') else x for x in output)\n            copy={k:self._copy(v) for k,v in kwargs.items()}\n            copy[\'encoder_hidden_states\']=self.negative\n            # Finish peer copies before launching either branch on another thread.\n            self.torch.cuda.synchronize(1)\n            self.pending=self.pool.submit(self._unconditional,copy)\n            return self.original(**kwargs)\n        self.primary.forward=forward\n        return self\n\n    def _copy(self,value):\n        if self.torch.is_tensor(value):return value.to(\'cuda:1\')\n        if isinstance(value,dict):return {k:self._copy(v) for k,v in value.items()}\n        if isinstance(value,(tuple,list)):return type(value)(self._copy(v) for v in value)\n        return value\n\n    def _unconditional(self,kwargs):\n        with self.torch.inference_mode(),self.torch.cuda.device(1):\n            context=getattr(self.secondary,\'cache_context\',None)\n            with context(\'uncond\') if context else nullcontext():output=self.secondary(**kwargs)\n            event=self.torch.cuda.Event();event.record(self.torch.cuda.current_stream(1))\n            return output,event\n\n    def __exit__(self,kind,*rest):\n        self.primary.forward=self.original\n        self.pool.shutdown(wait=True,cancel_futures=True)\n        unpaired=self.pending is not None;self.pending=None\n        if kind is None and (unpaired or self.calls==0):raise RuntimeError(\'Unpaired Wan CFG calls\')\n', 'video_attention.py': '"""Broadcast shared LTX prompt masks across Ulysses-sharded attention heads."""\nfrom functools import wraps\n\n\ndef broadcast_ltx_prompt_mask(mask,torch_module=None):\n    if mask is None or mask.ndim!=4 or mask.shape[1]==1:return mask\n    if torch_module is None:\n        import torch as torch_module\n    shared=mask[:,:1]\n    if not torch_module.equal(mask,shared.expand_as(mask)):\n        raise ValueError(\'LTX parallel attention requires a shared prompt mask\')\n    return shared\n\n\ndef enable_ltx_parallel_mask_broadcast():\n    # This module patch lives only inside the dedicated LTX rank process.\n    from diffusers.models.transformers import transformer_ltx\n    original=transformer_ltx.dispatch_attention_fn\n    if getattr(original,\'_cc_ltx_mask_broadcast\',False):return\n    @wraps(original)\n    def dispatch(*args,**kwargs):\n        if kwargs.get(\'parallel_config\') is not None:\n            kwargs[\'attn_mask\']=broadcast_ltx_prompt_mask(kwargs.get(\'attn_mask\'))\n        return original(*args,**kwargs)\n    dispatch._cc_ltx_mask_broadcast=True\n    transformer_ltx.dispatch_attention_fn=dispatch\n', 'video_schedule.py': '"""Preserve the distilled checkpoint schedule through LTX\'s pipeline defaults."""\n\n\ndef ltx_distilled_scheduler(config,trained_sigmas):\n    from diffusers import FlowMatchEulerDiscreteScheduler\n    trained=tuple(float(value) for value in trained_sigmas)\n    if len(trained)!=8 or trained[0]!=1.0 or not all(0<b<a<=1 for a,b in zip(trained,trained[1:])):\n        raise ValueError(\'Invalid distilled LTX schedule\')\n    class CCLTXDistilledScheduler(FlowMatchEulerDiscreteScheduler):\n        def set_timesteps(self,num_inference_steps=None,device=None,sigmas=None,mu=None,timesteps=None):\n            count=len(sigmas) if sigmas is not None else num_inference_steps\n            if timesteps is not None or count!=len(trained):\n                raise ValueError(\'Distilled LTX requires its trained eight-step sigma schedule\')\n            # LTXPipeline supplies a linear sigma array even when custom timesteps\n            # are requested. Replace that array at the scheduler boundary.\n            return super().set_timesteps(device=device,sigmas=list(trained),mu=mu)\n    return CCLTXDistilledScheduler.from_config(config,use_dynamic_shifting=False,shift=1.0)\n', 'video_context.py': '"""Persistent, two-process Ulysses runner for distilled LTX video sampling."""\nimport json\nimport hashlib\nimport os\nimport signal\nimport subprocess\nimport sys\nimport tempfile\nimport time\nimport uuid\nfrom pathlib import Path\n\n\ndef context_transformer_folder(folder,profile):\n    digest=hashlib.sha256(json.dumps(profile,sort_keys=True).encode()).hexdigest()[:16]\n    return Path(folder)/(\'context-transformer-\'+digest)\n\n\nclass LTXContextRunner:\n    def __init__(self,folder,profile,transformer,vae_config,scheduler_config,progress):\n        self.progress=progress;self.profile=profile;self.process=None;self.offset=0\n        self.temporary=tempfile.TemporaryDirectory(prefix=\'cc-video-cp-\',dir=\'/kaggle/temp\')\n        self.directory=Path(self.temporary.name)\n        self.weights=context_transformer_folder(folder,profile)\n        if not (self.weights/\'config.json\').is_file():\n            transformer.save_pretrained(self.weights,safe_serialization=True,max_shard_size=\'1GB\')\n        (self.directory/\'config.json\').write_text(json.dumps({\'weights\':str(self.weights),\n            \'vae\':dict(vae_config),\'scheduler\':dict(scheduler_config),\'profile\':profile}),encoding=\'utf-8\')\n\n    def start(self):\n        self.log=(self.directory/\'worker.log\').open(\'w\',encoding=\'utf-8\')\n        self.process=subprocess.Popen([sys.executable,\'-m\',\'torch.distributed.run\',\'--standalone\',\n            \'--nproc_per_node=2\',str(Path(__file__).resolve()),\'--worker\',str(self.directory)],\n            stdin=subprocess.DEVNULL,stdout=self.log,stderr=self.log,start_new_session=not bool(os.environ.get(\'CC_VIDEO_ACTOR\')))\n        deadline=time.monotonic()+240\n        while time.monotonic()<deadline:\n            self._check()\n            if all((self.directory/f\'ready-{i}\').exists() for i in range(2)):return self\n            time.sleep(.1)\n        self.close();raise TimeoutError(\'Video context workers did not become ready\')\n\n    def _check(self):\n        if self.process.poll() is not None:\n            detail=(self.directory/\'worker.log\').read_text(encoding=\'utf-8\',errors=\'replace\')[-2500:]\n            raise RuntimeError(\'Video context worker exited: \'+detail)\n        path=self.directory/\'worker.log\'\n        with path.open(encoding=\'utf-8\',errors=\'replace\') as source:\n            source.seek(self.offset)\n            while True:\n                line=source.readline()\n                if not line:break\n                if line.startswith(\'CC_CONTEXT_PROGRESS:\'):\n                    self.progress(json.loads(line.split(\':\',1)[1]))\n            self.offset=source.tell()\n\n    def generate(self,positive,mask,params):\n        import torch\n        job=uuid.uuid4().hex\n        torch.save({\'positive\':positive.cpu(),\'mask\':mask.cpu()},self.directory/(job+\'.input.pt\'))\n        temporary=self.directory/\'request.tmp\'\n        temporary.write_text(json.dumps(dict(params,job=job)),encoding=\'utf-8\')\n        temporary.replace(self.directory/\'request.json\')\n        deadline=time.monotonic()+14400\n        while time.monotonic()<deadline:\n            self._check()\n            output=self.directory/(job+\'.result.pt\')\n            if output.is_file():\n                result=torch.load(output,map_location=\'cpu\',weights_only=True)\n                footprints=[json.loads((self.directory/f\'{job}.rank-{i}.json\').read_text()) for i in range(2)]\n                result[\'worker_peak_gib\']=[item[\'peak_gpu_gib\'] for item in footprints]\n                result[\'worker_resident_gib\']=[item[\'resident_gpu_gib\'] for item in footprints]\n                (self.directory/(job+\'.input.pt\')).unlink();output.unlink()\n                return result\n            time.sleep(.1)\n        raise TimeoutError(\'Video sampling timed out\')\n\n    def _terminate(self,kind):\n        try:\n            if os.environ.get(\'CC_VIDEO_ACTOR\'):self.process.send_signal(kind)\n            else:os.killpg(self.process.pid,kind)\n        except ProcessLookupError:pass\n\n    def close(self):\n        if self.process is not None and self.process.poll() is None:\n            self._terminate(signal.SIGTERM)\n            try:self.process.wait(timeout=10)\n            except subprocess.TimeoutExpired:\n                self._terminate(signal.SIGKILL);self.process.wait(timeout=10)\n        if hasattr(self,\'log\'):self.log.close()\n        self.temporary.cleanup()\n\n\ndef worker(directory):\n    import torch\n    import torch.distributed as dist\n    from accelerate import init_empty_weights\n    from diffusers import (LTXPipeline,LTXVideoTransformer3DModel,AutoencoderKLLTXVideo,\n                           ContextParallelConfig)\n    from video_schedule import ltx_distilled_scheduler\n    rank=int(os.environ[\'LOCAL_RANK\']);device=torch.device(f\'cuda:{rank}\')\n    torch.cuda.set_device(device)\n    dist.init_process_group(backend=\'nccl\')\n    config=json.loads((directory/\'config.json\').read_text());p=config[\'profile\']\n    transformer=LTXVideoTransformer3DModel.from_pretrained(config[\'weights\'],\n        torch_dtype=torch.float16,device_map={\'\':rank},low_cpu_mem_usage=True,local_files_only=True)\n    from video_attention import enable_ltx_parallel_mask_broadcast\n    enable_ltx_parallel_mask_broadcast()\n    transformer.enable_parallelism(config=ContextParallelConfig(ulysses_degree=2))\n    transformer.set_attention_backend(\'native\')\n    with init_empty_weights():vae=AutoencoderKLLTXVideo.from_config(config[\'vae\'])\n    class DevicePipeline(LTXPipeline):\n        @property\n        def _execution_device(self):return device\n    pipeline=DevicePipeline(transformer=transformer,vae=vae,text_encoder=None,tokenizer=None,\n        scheduler=ltx_distilled_scheduler(config[\'scheduler\'],p[\'sigmas\']))\n    resident_gpu_gib=torch.cuda.memory_allocated(device)/2**30\n    (directory/f\'ready-{rank}\').touch();previous=None\n    try:\n        while True:\n            request=directory/\'request.json\'\n            if not request.exists():time.sleep(.05);continue\n            params=json.loads(request.read_text());job=params[\'job\']\n            if job==previous:time.sleep(.05);continue\n            previous=job\n            inputs=torch.load(directory/(job+\'.input.pt\'),map_location=\'cpu\',weights_only=True)\n            torch.cuda.reset_peak_memory_stats(device)\n            def progress(pipe,index,timestep,kwargs):\n                if not torch.isfinite(kwargs[\'latents\']).all():raise RuntimeError(\'Non-finite context latents\')\n                if rank==0:print(\'CC_CONTEXT_PROGRESS:\'+json.dumps({\'phase\':\'generating\',\'step\':index+1,\'steps\':params[\'steps\']}),flush=True)\n                return kwargs\n            generator=torch.Generator(device=device).manual_seed(params[\'seed\'])\n            result=pipeline(prompt_embeds=inputs[\'positive\'].to(device),\n                prompt_attention_mask=inputs[\'mask\'].to(device),guidance_scale=1,\n                height=p[\'height\'],width=p[\'width\'],num_frames=params[\'frames\'],\n                frame_rate=p[\'fps\'],num_inference_steps=params[\'steps\'],generator=generator,\n                callback_on_step_end=progress,output_type=\'latent\').frames\n            torch.cuda.synchronize(device)\n            (directory/f\'{job}.rank-{rank}.json\').write_text(json.dumps({\'peak_gpu_gib\':torch.cuda.max_memory_allocated(device)/2**30,\'resident_gpu_gib\':resident_gpu_gib}))\n            dist.barrier()\n            if rank==0:\n                target=directory/(job+\'.result.tmp\')\n                torch.save({\'latents\':result.cpu(),\'generator_state\':generator.get_state()},target)\n                target.replace(directory/(job+\'.result.pt\'))\n            del result,inputs\n    finally:dist.destroy_process_group()\n\n\nif __name__==\'__main__\':\n    if len(sys.argv)!=3 or sys.argv[1]!=\'--worker\':raise SystemExit(\'Use VideoRuntime\')\n    worker(Path(sys.argv[2]))\n', 'video_runtime.py': '"""GPU video runtime. Imported only on the remote Kaggle kernel."""\nimport gc\nimport json\nimport time\nfrom pathlib import Path\nimport torch\n\n\nclass VideoRuntime:\n    def __init__(self,model_id=\'wan21-t2v-1.3b\',acceleration=\'auto\',\n                 root=\'/kaggle/temp/cc-media-video\',progress=None,cache_handle=None,cache_token=None):\n        from video_registry import video_profile\n        if acceleration not in (\'auto\',\'split\'):raise ValueError(\'Invalid acceleration\')\n        self.profile=video_profile(model_id);self.acceleration=acceleration\n        self.folder=Path(root)/model_id;self.progress=progress or (lambda e:None)\n        self.pipe=None;self.encoder=None;self.replica=None;self.calls=0;self.disabled=False\n        self.cache_handle=cache_handle;self.cache_token=cache_token;self.cache=None;self.context=None\n\n    def emit(self,phase,**extra):self.progress(dict(phase=phase,**extra))\n\n    def prepare(self):\n        import torch\n        from transformers import BitsAndBytesConfig,T5EncoderModel,UMT5EncoderModel\n        from diffusers import (LTXPipeline,WanPipeline,\n                               WanTransformer3DModel,AutoencoderKLWan)\n        from video_assets import prepare_video_assets,video_cache_context\n        if torch.cuda.device_count()<2:raise RuntimeError(\'Select GPU T4 x2\')\n        start=time.perf_counter();p=self.profile\n        restored=False\n        if self.cache_handle and self.cache_token:\n            self.cache=video_cache_context(p,self.cache_handle,self.cache_token,self.progress)\n            try:\n                self.emit(\'cache-check\')\n                metadata=self.cache[\'cache_metadata\'](self.cache_handle,self.cache_token)\n                if metadata:\n                    self.cache[\'download_cache_assets\'](self.folder,self.cache_handle,self.cache_token,metadata)\n                    restored=True;self.cache[\'_QWEN_CACHE_STATUS\']=\'saved\'\n            except Exception:\n                self.cache[\'_QWEN_CACHE_STATUS\']=\'unavailable\';self.emit(\'cache-warning\')\n        if not restored:prepare_video_assets(p,self.folder,self.progress)\n        self.emit(\'loading\')\n        encoder_type=T5EncoderModel if p[\'family\']==\'ltx\' else UMT5EncoderModel\n        self.encoder=encoder_type.from_pretrained(self.folder/\'text_encoder\',\n            torch_dtype=torch.float16,device_map={\'\':1},low_cpu_mem_usage=True,\n            quantization_config=BitsAndBytesConfig(load_in_8bit=True),local_files_only=True)\n        if p[\'family\']==\'ltx\':\n            self.pipe=LTXPipeline.from_single_file(str(self.folder/p[\'checkpoint\']),\n                config=str(self.folder),text_encoder=self.encoder,torch_dtype=torch.float16,\n                local_files_only=True)\n            from video_schedule import ltx_distilled_scheduler\n            self.pipe.scheduler=ltx_distilled_scheduler(self.pipe.scheduler.config,p[\'sigmas\'])\n            self.pipe.transformer.to(\'cuda:0\');self.pipe.vae.to(device=\'cuda:0\',dtype=torch.float32)\n        else:\n            self.pipe_class=WanPipeline\n            self.transformer_type=WanTransformer3DModel\n            self.pipe=self.pipe_class.from_pretrained(self.folder,text_encoder=self.encoder,\n                transformer=self.load_transformer(0),\n                vae=AutoencoderKLWan.from_pretrained(self.folder/\'vae\',torch_dtype=torch.float32,\n                    local_files_only=True).to(\'cuda:0\'),torch_dtype=torch.float16,local_files_only=True)\n        if hasattr(self.pipe.vae,\'enable_tiling\'):self.pipe.vae.enable_tiling()\n        if hasattr(self.pipe.vae,\'enable_slicing\'):self.pipe.vae.enable_slicing()\n        self.load_seconds=time.perf_counter()-start\n        if self.cache and not restored:self.cache[\'save_private_cache\'](self.folder,self.encoder)\n        return self\n\n    def load_transformer(self,device):\n        import torch\n        from diffusers import BitsAndBytesConfig\n        kwargs={}\n        if self.profile[\'family\']==\'wan22\':kwargs[\'quantization_config\']=BitsAndBytesConfig(load_in_8bit=True)\n        return self.transformer_type.from_pretrained(self.folder/\'transformer\',\n            torch_dtype=torch.float16,device_map={\'\':device},low_cpu_mem_usage=True,\n            local_files_only=True,**kwargs)\n\n    @torch.inference_mode()\n    def first_frame(self,image):\n        """Decode the base64 first frame and center-crop it to the model size."""\n        import base64,io\n        from PIL import Image,ImageOps\n        picture=ImageOps.exif_transpose(Image.open(io.BytesIO(base64.b64decode(image))))\n        return ImageOps.fit(picture.convert(\'RGB\'),(self.profile[\'width\'],self.profile[\'height\']),\n            method=Image.Resampling.LANCZOS)\n\n    def image_pipeline(self):\n        """An image-to-video view of the loaded modules; no weights are copied."""\n        p=self.profile;pipe=self.pipe\n        if p[\'family\']==\'ltx\':\n            from diffusers import LTXImageToVideoPipeline\n            return LTXImageToVideoPipeline(scheduler=pipe.scheduler,vae=pipe.vae,text_encoder=pipe.text_encoder,\n                tokenizer=pipe.tokenizer,transformer=pipe.transformer)\n        from diffusers import WanImageToVideoPipeline\n        # Wan2.2 TI2V conditions on the first latent frame through expanded\n        # timesteps; it has no CLIP image encoder.\n        return WanImageToVideoPipeline(tokenizer=pipe.tokenizer,text_encoder=pipe.text_encoder,vae=pipe.vae,\n            scheduler=pipe.scheduler,transformer=pipe.transformer,expand_timesteps=True)\n\n    def restore_ltx_transformer(self):\n        """Stop context-parallel workers and load the single-GPU transformer."""\n        import torch\n        from diffusers import LTXVideoTransformer3DModel\n        from video_context import context_transformer_folder\n        from video_precision import move_video_encoder\n        self.context.close();self.context=None\n        self.pipe.register_modules(transformer=LTXVideoTransformer3DModel.from_pretrained(\n            context_transformer_folder(self.folder,self.profile),torch_dtype=torch.float16,\n            device_map={\'\':0},local_files_only=True))\n        move_video_encoder(self.encoder,\'cpu\');torch.cuda.empty_cache()\n\n    @torch.inference_mode()\n    def generate(self,prompt,seed=42,steps=None,frames=None,output=None,image=None,negative_prompt=None):\n        import torch\n        from video_registry import validate_video\n        from video_precision import move_video_encoder\n        from video_parallel import WanParallelCFG\n        p=self.profile;params=validate_video(prompt,seed,steps,p[\'id\'],frames,image,negative_prompt)\n        first=self.first_frame(image) if image is not None else None\n        if self.pipe is None:self.prepare()\n        for i in range(2):torch.cuda.reset_peak_memory_stats(i)\n        start=time.perf_counter();self.emit(\'encoding\')\n        # The 5B replica and UMT5 encoding workspace should not overlap on GPU 1.\n        if p[\'family\']==\'wan22\' and self.replica is not None:\n            move_video_encoder(self.replica,\'cpu\');torch.cuda.empty_cache()\n        move_video_encoder(self.encoder,\'cuda:1\')\n        encoded=self.pipe.encode_prompt(prompt=prompt,negative_prompt=negative_prompt or \'\',\n            do_classifier_free_guidance=p[\'guidance\']>1,device=torch.device(\'cuda:1\'))\n        if p[\'family\']==\'ltx\':positive,mask,negative,negative_mask=encoded\n        else:positive,negative=encoded[:2]\n        if not torch.isfinite(positive).all() or (negative is not None and not torch.isfinite(negative).all()):\n            raise RuntimeError(\'Non-finite video prompt embeddings\')\n        positive=positive.to(\'cuda:0\');negative=negative.to(\'cuda:0\') if negative is not None else None\n        torch.cuda.synchronize();encoding=time.perf_counter()-start\n        move_video_encoder(self.encoder,\'cpu\');torch.cuda.empty_cache()\n        if self.acceleration==\'split\' and self.replica is not None:\n            self.replica=None;gc.collect();torch.cuda.empty_cache()\n        replica_reused=self.replica is not None;context_reused=self.context is not None\n        replica_seconds=0;strategy=\'encoder_split\'\n        if p[\'family\']!=\'ltx\' and self.acceleration==\'auto\' and not self.disabled:\n            try:\n                if self.replica is None:\n                    tick=time.perf_counter();self.replica=self.load_transformer(1);replica_seconds=time.perf_counter()-tick\n                elif p[\'family\']==\'wan22\':move_video_encoder(self.replica,\'cuda:1\')\n                strategy=\'cfg_parallel\'\n            except (RuntimeError,ValueError):\n                self.replica=None;self.disabled=True;gc.collect();torch.cuda.empty_cache()\n        latents_finite=[]\n        def progress(pipeline,index,timestep,kwargs):\n            self.emit(\'generating\',step=index+1,steps=params[\'steps\'])\n            if index==params[\'steps\']-1:\n                latents_finite.append(bool(torch.isfinite(kwargs[\'latents\']).all().item()))\n                if not latents_finite[-1]:raise RuntimeError(\'Non-finite video latents\')\n            return kwargs\n        kwargs=dict(prompt_embeds=positive,negative_prompt_embeds=negative,\n            width=p[\'width\'],height=p[\'height\'],num_frames=params[\'frames\'],\n            num_inference_steps=params[\'steps\'],guidance_scale=p[\'guidance\'],\n            generator=torch.Generator(device=\'cuda:0\').manual_seed(seed),\n            callback_on_step_end=progress,output_type=\'np\')\n        if p[\'family\']==\'ltx\':kwargs.update(decode_timestep=.05,decode_noise_scale=.025,frame_rate=p[\'fps\'],\n            prompt_attention_mask=mask.to(\'cuda:0\'),negative_prompt_attention_mask=negative_mask)\n        # Image-to-video uses the same modules through an image pipeline view.\n        target=self.pipe\n        if first is not None:\n            kwargs[\'image\']=first\n            if p[\'family\']==\'ltx\' and self.context is not None:self.restore_ltx_transformer()\n            target=self.image_pipeline()\n        self.pipe.register_modules(text_encoder=None);target.register_modules(text_encoder=None)\n        sampling=time.perf_counter();fallback=None;cfg_pairs=0\n        original_decode=self.pipe.vae.decode\n        vae_dtype=self.pipe.vae.dtype\n        def decode(*args,**kwargs):\n            self.emit(\'decoding\')\n            # LTXImageToVideoPipeline decodes fp16 latents and an fp16 decode\n            # timestep with the fp32 VAE (its text-to-video pipeline casts them).\n            args=[a.to(dtype=vae_dtype) if torch.is_tensor(a) and a.is_floating_point() else a for a in args]\n            return original_decode(*args,**kwargs)\n        self.pipe.vae.decode=decode\n        original_encode=self.pipe.vae.encode\n        if first is not None and p[\'family\']==\'ltx\':\n            # LTXImageToVideoPipeline passes the first frame in the prompt dtype\n            # (fp16) while this runtime keeps the VAE in fp32.\n            vae=self.pipe.vae\n            self.pipe.vae.encode=lambda x,*args,**kwargs:original_encode(x.to(dtype=vae.dtype),*args,**kwargs)\n        try:\n            if first is not None and strategy!=\'cfg_parallel\':\n                # LTX context parallelism is text-to-video only; image-to-video\n                # runs the single-GPU path with the encoder already moved aside.\n                result=target(**kwargs)\n            elif p[\'family\']==\'ltx\' and self.acceleration==\'auto\' and not self.disabled:\n                from video_context import LTXContextRunner\n                if self.context is None:\n                    runner=None;original=self.pipe.transformer\n                    try:\n                        runner=LTXContextRunner(self.folder,p,original,self.pipe.vae.config,\n                            self.pipe.scheduler.config,self.progress)\n                        original.to(\'cpu\');self.pipe.register_modules(transformer=None)\n                        move_video_encoder(self.encoder,\'cuda:1\');gc.collect();torch.cuda.empty_cache()\n                        self.context=runner.start()\n                    except Exception as error:\n                        print(\'CC_VIDEO_CONTEXT_ERROR:\'+str(error),flush=True)\n                        if runner is not None:runner.close()\n                        self.pipe.register_modules(transformer=original.to(\'cuda:0\'))\n                        self.disabled=True;fallback=type(error).__name__\n                    finally:del original\n                if self.context is not None:\n                    try:packed=self.context.generate(positive,mask,params)\n                    except (RuntimeError,ValueError,TimeoutError) as error:\n                        print(\'CC_VIDEO_CONTEXT_ERROR:\'+str(error),flush=True)\n                        self.context.close();self.context=None;self.disabled=True;fallback=type(error).__name__\n                        from diffusers import LTXVideoTransformer3DModel\n                        from video_context import context_transformer_folder\n                        self.pipe.register_modules(transformer=LTXVideoTransformer3DModel.from_pretrained(\n                            context_transformer_folder(self.folder,p),torch_dtype=torch.float16,\n                            device_map={\'\':0},local_files_only=True))\n                        move_video_encoder(self.encoder,\'cpu\');torch.cuda.empty_cache()\n                        result=self.pipe(**kwargs)\n                    else:\n                        generator=torch.Generator(device=\'cuda:0\');generator.set_state(packed[\'generator_state\'])\n                        decoded=self.decode_context_latents(packed[\'latents\'].to(\'cuda:0\'),params[\'frames\'],generator)\n                        from types import SimpleNamespace\n                        result=SimpleNamespace(frames=decoded);strategy=\'context_parallel\'\n                else:result=self.pipe(**kwargs)\n            elif strategy==\'cfg_parallel\':\n                try:\n                    router=WanParallelCFG(self.pipe.transformer,self.replica,negative)\n                    try:\n                        with router:result=target(**kwargs)\n                        cfg_pairs=router.calls\n                    finally:del router\n                except RuntimeError as error:\n                    if not isinstance(error,torch.cuda.OutOfMemoryError):raise\n                    fallback=type(error).__name__;self.replica=None;self.disabled=True\n                if fallback:\n                    gc.collect();torch.cuda.empty_cache();strategy=\'encoder_split\'\n                    kwargs[\'generator\']=torch.Generator(device=\'cuda:0\').manual_seed(seed)\n                    result=target(**kwargs)\n            else:\n                if self.context is not None:\n                    self.context.close();self.context=None\n                    from diffusers import LTXVideoTransformer3DModel\n                    from video_context import context_transformer_folder\n                    self.pipe.register_modules(transformer=LTXVideoTransformer3DModel.from_pretrained(\n                        context_transformer_folder(self.folder,p),torch_dtype=torch.float16,\n                        device_map={\'\':0},local_files_only=True))\n                result=self.pipe(**kwargs)\n        finally:\n            self.pipe.vae.decode=original_decode\n            self.pipe.vae.__dict__.pop(\'encode\',None)\n            self.pipe.register_modules(text_encoder=self.encoder)\n            if target is not self.pipe:target.register_modules(text_encoder=self.encoder)\n        torch.cuda.synchronize();sampling_seconds=time.perf_counter()-sampling\n        import numpy as np\n        video=np.asarray(result.frames[0])\n        if not np.isfinite(video).all() or float(video.std())<.001:raise RuntimeError(\'Invalid or blank video\')\n        self.emit(\'exporting\');from diffusers.utils import export_to_video\n        output=Path(output or \'/kaggle/temp/cc-media-video-result.mp4\');output.parent.mkdir(parents=True,exist_ok=True)\n        export_to_video(video,str(output),fps=p[\'fps\'])\n        self.calls+=1\n        peaks=[torch.cuda.max_memory_allocated(i)/2**30 for i in range(2)]\n        if strategy==\'context_parallel\':\n            # Worker weights remain resident while the parent encodes or decodes.\n            peaks=[max(peaks[i]+packed[\'worker_resident_gib\'][i],\n                packed[\'worker_peak_gib\'][i]+torch.cuda.memory_allocated(i)/2**30) for i in range(2)]\n        return output,dict(model_id=p[\'id\'],media_type=\'video\',width=p[\'width\'],height=p[\'height\'],\n            frames=params[\'frames\'],fps=p[\'fps\'],steps=params[\'steps\'],seed=seed,\n            encoding_seconds=round(encoding,3),generation_seconds=round(sampling_seconds,3),\n            request_seconds=round(time.perf_counter()-start,3),loading_seconds=round(self.load_seconds,3),\n            model_reused=self.calls>1,model_load_count=1,generation_number=self.calls,\n            parallel_strategy=strategy,acceleration_mode=self.acceleration,cfg_pairs=cfg_pairs,\n            replica_load_seconds=round(replica_seconds,3),dual_model_reused=(replica_reused if strategy==\'cfg_parallel\' else context_reused if strategy==\'context_parallel\' else False),\n            acceleration_fallback=fallback,peak_gpu_gib=[round(v,3) for v in peaks],image_to_video=first is not None)\n\n    def decode_context_latents(self,latents,frames,generator):\n        import torch\n        pipe=self.pipe;p=self.profile\n        latents=pipe._unpack_latents(latents,(frames-1)//pipe.vae_temporal_compression_ratio+1,\n            p[\'height\']//pipe.vae_spatial_compression_ratio,p[\'width\']//pipe.vae_spatial_compression_ratio,\n            pipe.transformer_spatial_patch_size,pipe.transformer_temporal_patch_size)\n        latents=pipe._denormalize_latents(latents,pipe.vae.latents_mean,pipe.vae.latents_std,pipe.vae.config.scaling_factor)\n        timestep=None\n        if pipe.vae.config.timestep_conditioning:\n            from diffusers.utils.torch_utils import randn_tensor\n            noise=randn_tensor(latents.shape,generator=generator,device=latents.device,dtype=latents.dtype)\n            latents=.975*latents+.025*noise\n            timestep=torch.tensor([.05],device=latents.device,dtype=latents.dtype)\n        decoded=pipe.vae.decode(latents.to(pipe.vae.dtype),timestep,return_dict=False)[0]\n        return pipe.video_processor.postprocess_video(decoded,output_type=\'np\')\n\n    def close(self):\n        if self.context is not None:self.context.close();self.context=None\n        self.pipe=None;self.encoder=None;self.replica=None\n        self.cache=None\n        gc.collect()\n        import torch\n        torch.cuda.empty_cache()\n', 'video_process.py': '"""Persistent video worker in a separate Python environment on Kaggle."""\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport select\nimport signal\nimport subprocess\nimport sys\nimport time\nimport uuid\n\n\nclass VideoProcess:\n    def __init__(self, model_id, acceleration=\'auto\', progress=None, cache_handle=None,\n                 cache_token=None, install=True, environment=\'/kaggle/temp/cc-media-video-environment\'):\n        from video_registry import video_profile\n        self.profile=video_profile(model_id);self.progress=progress or (lambda event:None)\n        self.acceleration=acceleration;self.process=None;self.buffer=b\'\'\n        requirements=Path(__file__).with_name(\'requirements-video-remote.txt\')\n        environment=Path(environment);python=environment/\'bin/python\'\n        if install:\n            if not python.exists():\n                subprocess.run([sys.executable,\'-m\',\'venv\',\'--without-pip\',\'--system-site-packages\',str(environment)],check=True)\n            digest=hashlib.sha256(requirements.read_bytes()).hexdigest()\n            stamp=environment/\'cc-requirements.sha256\'\n            if not stamp.exists() or stamp.read_text()!=digest:\n                self.progress({\'phase\':\'installing\'})\n                subprocess.run([str(python),\'-m\',\'pip\',\'install\',\'-q\',\'-r\',str(requirements)],check=True,timeout=1200)\n                stamp.write_text(digest)\n        elif not python.exists():python=Path(sys.executable)\n        env=dict(os.environ,USE_TF=\'0\',USE_FLAX=\'0\',TOKENIZERS_PARALLELISM=\'false\',CC_VIDEO_ACTOR=\'1\')\n        self.process=subprocess.Popen([str(python),\'-u\',str(Path(__file__).resolve()),\'--worker\'],\n            stdin=subprocess.PIPE,stdout=subprocess.PIPE,stderr=None,start_new_session=True,env=env)\n        try:self.request(\'configure\',model_id=model_id,acceleration=acceleration,\n                         cache_handle=cache_handle,cache_token=cache_token)\n        except BaseException:self.close();raise\n\n    def alive(self):return self.process is not None and self.process.poll() is None\n\n    def request(self,action,**params):\n        if not self.alive():raise RuntimeError(\'Video process stopped\')\n        request_id=uuid.uuid4().hex\n        self.process.stdin.write((json.dumps(dict(id=request_id,action=action,**params))+\'\\n\').encode())\n        self.process.stdin.flush();deadline=time.monotonic()+14400\n        while time.monotonic()<deadline:\n            if b\'\\n\' not in self.buffer:\n                if not select.select([self.process.stdout],[],[],min(5,max(0,deadline-time.monotonic())))[0]:\n                    if not self.alive():raise RuntimeError(\'Video process exited\')\n                    continue\n                chunk=os.read(self.process.stdout.fileno(),65536)\n                if not chunk:raise RuntimeError(\'Video process output closed\')\n                self.buffer+=chunk\n                if len(self.buffer)>2**20:raise RuntimeError(\'Video response exceeded limit\')\n                continue\n            line,self.buffer=self.buffer.split(b\'\\n\',1)\n            if not line.startswith(b\'CC_VIDEO_PROCESS:\'):continue\n            message=json.loads(line.split(b\':\',1)[1])\n            if message.get(\'id\')!=request_id:raise RuntimeError(\'Video response mismatch\')\n            if \'progress\' in message:self.progress(message[\'progress\']);continue\n            if \'error\' in message:raise RuntimeError(message[\'error\'])\n            return message.get(\'result\')\n        self.close();raise TimeoutError(\'Video request timed out\')\n\n    def generate(self,prompt,seed=42,steps=None,frames=None,output=None,image=None,negative_prompt=None):\n        # The first frame travels as base64 on the worker\'s JSON stdin line.\n        extra={k:v for k,v in ((\'image\',image),(\'negative_prompt\',negative_prompt)) if v is not None}\n        result=self.request(\'generate\',prompt=prompt,seed=seed,steps=steps,frames=frames,\n                            output=output,acceleration=self.acceleration,**extra)\n        return Path(result[\'path\']),result[\'stats\']\n\n    def close(self):\n        process=self.process;self.process=None\n        if process is None:return\n        try:\n            if process.poll() is None:\n                process.stdin.write((json.dumps({\'id\':uuid.uuid4().hex,\'action\':\'close\'})+\'\\n\').encode());process.stdin.flush()\n                try:process.wait(timeout=15)\n                except subprocess.TimeoutExpired:\n                    os.killpg(process.pid,signal.SIGTERM)\n                    try:process.wait(timeout=10)\n                    except subprocess.TimeoutExpired:os.killpg(process.pid,signal.SIGKILL);process.wait(timeout=10)\n        except (BrokenPipeError,ProcessLookupError):pass\n        finally:\n            process.stdin.close();process.stdout.close()\n\n\ndef worker():\n    import contextlib\n    import traceback\n    wire=sys.stdout;runtime=None;request_id=None\n    def send(**data):\n        wire.write(\'CC_VIDEO_PROCESS:\'+json.dumps(dict(id=request_id,**data))+\'\\n\');wire.flush()\n    try:\n        for line in sys.stdin:\n            command=json.loads(line);request_id=command.pop(\'id\');action=command.pop(\'action\')\n            try:\n                if action==\'close\':break\n                with contextlib.redirect_stdout(sys.stderr):\n                    if action==\'configure\':\n                        from video_runtime import VideoRuntime\n                        runtime=VideoRuntime(**command,progress=lambda event:send(progress=event)).prepare()\n                        result={\'ready\':True}\n                    elif action==\'generate\' and runtime is not None:\n                        runtime.acceleration=command.pop(\'acceleration\')\n                        path,stats=runtime.generate(**command);result={\'path\':str(path),\'stats\':stats}\n                    else:raise ValueError(\'Unknown video command\')\n                send(result=result)\n            except Exception as error:\n                traceback.print_exc(file=sys.stderr)\n                # Include wrapped causes: libraries often re-raise a generic error.\n                parts,current=[],error\n                while current is not None and len(parts)<6:\n                    parts.append(type(current).__name__+\': \'+str(current)[:300]);current=current.__cause__ or current.__context__\n                send(error=\' <- \'.join(parts))\n    finally:\n        if runtime is not None:\n            with contextlib.redirect_stdout(sys.stderr):runtime.close()\n\n\nif __name__==\'__main__\' and \'--worker\' in sys.argv:worker()\n', 'requirements-video-remote.txt': '# Installed on the remote GPU only; local clients need no Torch or CUDA.\ndiffusers==0.42.1\ntransformers==5.22.2\naccelerate==1.12.0\nbitsandbytes==0.50.2\nsafetensors\nsentencepiece\nimageio-ffmpeg\nftfy>=6.3\nimageio>=2.37\n'}.items():(modules/name).write_text(source,encoding='utf-8')
sys.path.insert(0,str(modules))
print('CC Media Studio video ready; first generation installs an isolated environment')


In [ ]:
import ipywidgets as widgets
from IPython.display import Video,display,clear_output
from video_registry import VIDEO_MODELS
from video_process import VideoProcess
model=widgets.Dropdown(options=[(p['name'],p['id']) for p in VIDEO_MODELS.values()],value='wan21-t2v-1.3b',description='Model')
prompt=widgets.Textarea(value='An adult anime catgirl reading by a sunlit window, hair moving gently in the breeze. Slow camera movement.',description='Prompt',layout=widgets.Layout(width='95%',height='100px'))
frames=widgets.Dropdown(options=[9,17,33,49,65,81],value=33,description='Frames')
steps=widgets.IntSlider(value=30,min=1,max=50,description='Steps')
seed=widgets.IntText(value=42,description='Seed')
acceleration=widgets.Dropdown(options=[('Auto / 双卡优先','auto'),('Encoder split / 编码分卡','split')],value='auto',description='GPU')
negative=widgets.Textarea(value='',description='Negative',placeholder='Wan only / 仅 Wan',layout=widgets.Layout(width='95%',height='60px'))
first_frame=widgets.FileUpload(accept='image/png,image/jpeg,image/webp',multiple=False,description='First frame / 首帧')
frame_hint=widgets.HTML()
button=widgets.Button(description='Generate / 生成',button_style='primary')
output=widgets.Output();runtime=None
def selected(change=None):
    p=VIDEO_MODELS[model.value];steps.value=p['default_steps'];steps.disabled=bool(p['fixed_steps'])
    negative.layout.display='' if p['guidance']>1 else 'none'
    first_frame.layout.display='' if p['image_to_video'] else 'none'
    frame_hint.value='<span style="color:#777">'+('Optional first frame, center-cropped to %d × %d / 可选首帧，居中裁剪' % (p['width'],p['height']) if p['image_to_video'] else 'Text-to-video only / 仅文生视频')+'</span>'
model.observe(selected,names='value');selected()
def frame_data():
    value=first_frame.value
    if not value or not VIDEO_MODELS[model.value]['image_to_video']:return None
    import base64
    item=value[0] if isinstance(value,(tuple,list)) else next(iter(value.values()))
    return base64.b64encode(bytes(item['content'])).decode('ascii')
def generate(_):
    global runtime
    button.disabled=True
    try:
        with output:
            clear_output(wait=True)
            if runtime is not None and runtime.profile['id']!=model.value:runtime.close();runtime=None
            if runtime is None:runtime=VideoProcess(model.value,acceleration=acceleration.value,progress=lambda event:print(event,flush=True))
            runtime.acceleration=acceleration.value
            path,usage=runtime.generate(prompt.value,seed=seed.value,steps=steps.value,frames=frames.value,output='/kaggle/working/cc-media-video.mp4',
                image=frame_data(),negative_prompt=(negative.value or None) if VIDEO_MODELS[model.value]['guidance']>1 else None)
            display(Video(str(path),embed=True));print(usage)
    finally:button.disabled=False
button.on_click(generate)
display(widgets.VBox([model,prompt,negative,first_frame,frame_hint,widgets.HBox([frames,steps]),widgets.HBox([seed,acceleration]),button,output]))
